In [95]:
import pandas as pd
import sqlite3 as sq
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
import requests
pd.set_option('display.max_rows', 1000); pd.set_option('display.max_columns', 1000); pd.set_option('display.width', 1000)
pd.options.mode.chained_assignment = None
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

In [96]:
twentyfive = pd.read_csv('../data/urbansim/Run 38 - Pipeline 5_9 (county, block) - 2025.csv')
thirty = pd.read_csv('../data/urbansim/Run 38 - Pipeline 5_9 (county, block) - 2030.csv')
forty = pd.read_csv('../data/urbansim/Run 38 - Pipeline 5_9 (county, block) - 2040.csv')
fifty = pd.read_csv('../data/urbansim/Run 38 - Pipeline 5_9 (county, block) - 2050.csv')
dfs = [twentyfive, thirty, forty, fifty]
data = pd.concat(dfs)
data = data.loc[data['geo_level'] == 'county']
data = data[['year', 'geo_level_id', 'household_population', 'sum_total_households', 'sum_total_jobs']]
data.columns

Index(['year', 'geo_level_id', 'household_population', 'sum_total_households', 'sum_total_jobs'], dtype='object')

In [97]:
data['geo_level_id'].unique()

array([47083, 47119, 47169, 47085, 47189, 47125, 47165, 47161, 47043,
       47147, 47037, 47149, 47021, 47187], dtype=int64)

In [98]:
fipsdict = {47083: 'Houston',
            47119: 'Maury', 
            47169: 'Trousdale', 
            47085: 'Humphreys',
            47189: 'Wilson',
            47125: 'Montgomery',
            47165: 'Sumner',
            47161: 'Stewart',
            47043: 'Dickson',
            47147: 'Robertson',
            47037: 'Davidson',
            47149: 'Rutherford',
            47021: 'Cheatham',
            47187: 'Williamson'}
colsdict = {'household_population': 'HH Population', 
            'sum_total_households': 'Households', 
            'sum_total_jobs': 'Jobs', 
            'sum_total_units': 'Housing Units'
           }

In [99]:
data['County'] = data['geo_level_id'].map(fipsdict)
data = data.drop(columns = ['geo_level_id'])
data = data.rename(columns = colsdict)
data['year'] = data['year'].astype(str)
data.head()

,year,HH Population,Households,Jobs,County
36642,2025,7306.304199,3105.0,2700.0,Houston
36643,2025,107712.632812,44469.0,61774.0,Maury
36644,2025,9663.598633,3790.0,3712.0,Trousdale
36645,2025,15657.616211,6779.0,9204.0,Humphreys
36646,2025,165893.625000,65002.0,95641.0,Wilson


In [100]:
controls = pd.read_excel('../data/urbansim/County_Controls_2023_2050_temptilloffshared.xlsx', sheet_name = 'in')
controls = controls[['County', 'Category', 2023, 2025, 2030, 2040, 2050]]
controls['Category'].unique()

array(['Total Population', 'HH Population', 'GQ Population', 'Households',
       'Industrial', 'Office', 'Service', 'Other', 'Education',
       'Food Services', 'Government', 'Medical', 'Retail'], dtype=object)

In [101]:
controls = controls.loc[~controls['Category'].isin(['GQ Population', 'Total Population'])]
controls.head(2)

,County,Category,2023,2025,2030,2040,2050
14,Cheatham,HH Population,41924,42490,43624,46079,48725
15,Davidson,HH Population,688602,695651,717448,749125,777976


In [102]:
controls = controls.melt(id_vars = ['County', 'Category'], var_name = 'year')
controls = controls.pivot(columns = 'Category', index = ['County', 'year'], values = 'value').reset_index(drop = False)
controls = controls.rename_axis(None, axis=1)

In [103]:
controls.head()

,County,year,Education,Food Services,Government,HH Population,Households,Industrial,Medical,Office,Other,Retail,Service
0,Cheatham,2023,525,982,1216,41924,16125,4280,830,3720,2508,1493,1739
1,Cheatham,2025,541,1059,1216,42490,16494,4275,838,3730,2498,1477,1746
2,Cheatham,2030,595,1273,1187,43624,17272,4405,883,3948,2559,1500,1805
3,Cheatham,2040,693,1779,1126,46079,18545,4725,964,4305,2680,1530,1890
4,Cheatham,2050,778,2415,1047,48725,19969,5130,1025,4596,2814,1555,1946


In [104]:
thelist = [controls['Education'], controls['Food Services'], controls['Government'], controls['Industrial'], controls['Medical'], 
               controls['Office'], controls['Other'], controls['Retail'], controls['Service']]
controls['Jobs'] = sum(thelist)
controls = controls.drop(columns = ['Education', 'Food Services', 'Government', 'Industrial', 'Medical', 'Office', 'Other', 'Retail', 'Service'])
controls.head()

,County,year,HH Population,Households,Jobs
0,Cheatham,2023,41924,16125,17293
1,Cheatham,2025,42490,16494,17380
2,Cheatham,2030,43624,17272,18155
3,Cheatham,2040,46079,18545,19692
4,Cheatham,2050,48725,19969,21306


In [105]:
jobseqcheck = controls.loc[controls['year'] == 2023]
jobseqcheck = jobseqcheck[['County', 'Jobs']]
controls = controls.loc[controls['year'] != 2023]

In [106]:
data.head()

,year,HH Population,Households,Jobs,County
36642,2025,7306.304199,3105.0,2700.0,Houston
36643,2025,107712.632812,44469.0,61774.0,Maury
36644,2025,9663.598633,3790.0,3712.0,Trousdale
36645,2025,15657.616211,6779.0,9204.0,Humphreys
36646,2025,165893.625000,65002.0,95641.0,Wilson


In [107]:
data = data.set_index(['County', 'year']).add_suffix('_UrSim').reset_index(drop = False)

In [108]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56 entries, 0 to 55
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   County               56 non-null     object 
 1   year                 56 non-null     object 
 2   HH Population_UrSim  56 non-null     float64
 3   Households_UrSim     56 non-null     float64
 4   Jobs_UrSim           56 non-null     float64
dtypes: float64(3), object(2)
memory usage: 2.3+ KB


In [109]:
data['year'] = data['year'].astype(int)

In [110]:
controls.info()

<class 'pandas.core.frame.DataFrame'>
Index: 56 entries, 1 to 69
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   County         56 non-null     object
 1   year           56 non-null     int64 
 2   HH Population  56 non-null     int64 
 3   Households     56 non-null     int64 
 4   Jobs           56 non-null     int64 
dtypes: int64(4), object(1)
memory usage: 2.6+ KB


In [111]:
df = data.merge(controls, on = ['County', 'year'])

In [112]:
df.head()

,County,year,HH Population_UrSim,Households_UrSim,Jobs_UrSim,HH Population,Households,Jobs
0,Houston,2025,7306.304199,3105.0,2700.0,8285,3174,2700
1,Maury,2025,107712.632812,44469.0,61774.0,111634,44469,61774
2,Trousdale,2025,9663.598633,3790.0,3712.0,10022,3912,3712
3,Humphreys,2025,15657.616211,6779.0,9204.0,18883,7013,9204
4,Wilson,2025,165893.625000,65002.0,95641.0,167282,65135,95641


In [113]:
df['Households_Diff'] = abs((df['Households'] - df['Households_UrSim'])/df['Households_UrSim'])
df['HH Population_Diff'] = abs((df['HH Population'] - df['HH Population_UrSim'])/df['HH Population_UrSim'])
df['Jobs_Diff'] = abs((df['Jobs'] - df['Jobs_UrSim'])/df['Jobs_UrSim'])

In [114]:
df.head()

,County,year,HH Population_UrSim,Households_UrSim,Jobs_UrSim,HH Population,Households,Jobs,Households_Diff,HH Population_Diff,Jobs_Diff
0,Houston,2025,7306.304199,3105.0,2700.0,8285,3174,2700,0.022222,0.133952,0.0
1,Maury,2025,107712.632812,44469.0,61774.0,111634,44469,61774,0.000000,0.036406,0.0
2,Trousdale,2025,9663.598633,3790.0,3712.0,10022,3912,3712,0.032190,0.037088,0.0
3,Humphreys,2025,15657.616211,6779.0,9204.0,18883,7013,9204,0.034518,0.205995,0.0
4,Wilson,2025,165893.625000,65002.0,95641.0,167282,65135,95641,0.002046,0.008369,0.0


In [115]:
df.to_csv('../data/controls/comp.csv', index = False)

In [116]:
jobseqcheck.head()

,County,Jobs
0,Cheatham,17293
5,Davidson,753486
10,Dickson,28739
15,Houston,2662
20,Humphreys,9018


In [117]:
jobseqcheck['County'] = jobseqcheck['County'] + ' County, Tennessee'
jobseqcheck = jobseqcheck.rename(columns = {'Jobs': 'Jobs 2023 Control'})

In [118]:
jeq = pd.read_csv('../data/urbansim/JobsEQ_2023EMP_County.csv')

In [119]:
jeq.head()

,County,Jobs 2023 JobsEQ
0,"Cheatham County, Tennessee",11073.06917
1,"Davidson County, Tennessee",583103.60020
2,"Dickson County, Tennessee",20849.30289
3,"Houston County, Tennessee",1820.43863
4,"Humphreys County, Tennessee",6828.67318


In [120]:
emp = jobseqcheck.merge(jeq, on = ['County'])

In [121]:
emp

,County,Jobs 2023 Control,Jobs 2023 JobsEQ
0,"Cheatham County, Tennessee",17293,11073.069170
1,"Davidson County, Tennessee",753486,583103.600200
2,"Dickson County, Tennessee",28739,20849.302890
3,"Houston County, Tennessee",2662,1820.438630
4,"Humphreys County, Tennessee",9018,6828.673180
5,"Maury County, Tennessee",60244,45372.359650
6,"Montgomery County, Tennessee",96934,68407.418440
7,"Robertson County, Tennessee",38555,26397.019480
8,"Rutherford County, Tennessee",211518,157033.990100
9,"Stewart County, Tennessee",4999,3523.965237


In [122]:
emp['Diff'] = emp['Jobs 2023 JobsEQ'] - emp['Jobs 2023 Control']
emp['Diff %'] = (emp['Jobs 2023 JobsEQ'] - emp['Jobs 2023 Control'])/emp['Jobs 2023 Control']

In [123]:
emp.to_csv('../data/controls/jobseqempcomp.csv', index = False)